# Laboratory 07 — The second law and heat engines

In this laboratory you will build heat engines, measure their efficiencies, and try — and
fail — to design one that beats the Carnot bound.

The route: build a Carnot engine and read its ledger (Part 1); try to beat the bound, and
audit the impossible machines of the module's proofs (Part 2); run it backwards (Part 3);
find where lost work goes (Part 4); meet entropy the Clausius way (Part 5); test a real
engine cycle (Part 6); and explore freely (Part 7).

Work through it in order. Where the notebook asks you to predict, write your prediction in
the cell provided **before** running the next cell. A prediction you have committed to is the
only reliable way to discover that you were wrong.

## Model specification

| | |
|---|---|
| **System** | a fixed amount of ideal gas, $N$ particles with $f$ quadratic degrees of freedom |
| **Dynamics** | quasistatic strokes joined into a closed loop in the $P$–$V$ plane |
| **Boundary** | frictionless piston; wall switched between diathermal and adiabatic |
| **Ensemble** | not applicable — this is thermodynamics, nothing counts microstates |
| **Ignored** | friction, piston mass, gas non-ideality, leaks through the adiabats, the time a stroke takes |
| **Valid when** | strokes are slow compared with the relaxation time; reservoirs are large enough not to change temperature |
| **Failure modes** | finite-rate operation; regenerators, which store heat inside the engine between strokes; a working substance near condensation |

All the physics lives in `thermolab.engines` — open it and read it. Nothing in this course is
hidden inside a framework.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import engines, processes
from thermolab.constants import K_B
from thermolab.validation import relative_error

T_HOT = 600.0   # K — the hot reservoir
T_COLD = 300.0  # K — the cold reservoir
N_PARTICLES = 1000
V_START = 1.0e-3  # m^3

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

print(f"Carnot bound between {T_HOT:.0f} K and {T_COLD:.0f} K: "
      f"{engines.carnot_efficiency(T_HOT, T_COLD):.6f}")

## Part 1 — Build a Carnot engine and look at it

Four strokes: expand in contact with the hot reservoir, expand with the heat shut off,
compress against the cold reservoir, compress with the heat shut off again. The loop closes,
and the area it encloses is the work delivered.

In [ ]:
cycle = engines.carnot_cycle(N_PARTICLES, T_HOT, T_COLD, V_START, expansion_ratio=2.5)

fig, (plane, bars) = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [1.4, 1]})
for stroke, colour in zip(cycle.strokes,
                          ["#dc2626", "#94a3b8", "#2563eb", "#94a3b8"], strict=True):
    path = stroke.process.quasistatic_path
    plane.plot(path.volumes * 1e3, path.pressures, lw=2.4, color=colour)
plane.set_xlabel("volume (L)")
plane.set_ylabel("pressure (Pa)")
plane.set_title("the cycle")

bars.bar(["heat in", "work out", "heat dumped"],
         [cycle.heat_absorbed, cycle.work_output, cycle.heat_rejected],
         color=["#dc2626", "#0f172a", "#2563eb"])
bars.set_ylabel("energy per cycle (J)")
bars.set_title("the ledger")
plt.tight_layout()
plt.show()

print(f"heat absorbed   Q_h = {cycle.heat_absorbed:.4e} J")
print(f"work delivered  W   = {cycle.work_output:.4e} J")
print(f"heat rejected   Q_c = {cycle.heat_rejected:.4e} J")
print(f"efficiency          = {cycle.efficiency:.6f}")
print(f"Carnot bound        = {cycle.carnot_bound:.6f}")

### The ledger, stroke by stroke

The bars are totals. Here is the same cycle one stroke at a time, in the course's sign
convention: heat into the gas and work done on it both count as positive. Three columns are
worth watching.

- The two adiabats' works are equal and opposite, so they cancel — the whole net work comes
  from the isotherms.
- The gas's entropy change sums to zero round the loop, because entropy is a function of state
  and the gas ends where it began.
- The last column divides each heat by the temperature of the *reservoir* it crossed to or
  from. Its sum is the Clausius sum of the module page, and for this reversible cycle it is
  zero as well.

In [ ]:
header = (f"{'stroke':<18}{'Q (J)':>12}{'W_on (J)':>12}{'dU (J)':>12}"
          f"{'dS_gas (J/K)':>15}{'Q/T_res (J/K)':>15}")
print(header)
print("-" * len(header))
for stroke in cycle.strokes:
    process = stroke.process
    q_over_t = (process.heat / stroke.reservoir_temperature
                if stroke.reservoir_temperature is not None else 0.0)
    print(f"{process.label:<18}{process.heat:>12.3e}{process.work_on_gas:>12.3e}"
          f"{process.internal_energy_change:>12.3e}"
          f"{engines.entropy_change_of_gas(process):>15.3e}{q_over_t:>15.3e}")
print("-" * len(header))
gas_entropy = sum(engines.entropy_change_of_gas(s.process) for s in cycle.strokes)
print(f"{'round the loop':<18}{sum(s.process.heat for s in cycle.strokes):>12.3e}"
      f"{cycle.net_work_on_gas:>12.3e}{cycle.internal_energy_drift:>12.3e}"
      f"{gas_entropy:>15.3e}{cycle.clausius_sum:>15.3e}")
print("\n(the zeros in the last row are zero to rounding: compare them with the rows above)")

Notice that the work bar is exactly half the heat-in bar, and that the heat-dumped bar is the
other half. The engine is *perfect* — every stroke is reversible — and it still throws away
half of what it took in.

### Predict

Before running the next cell: you are about to build the same engine out of a different gas
(monatomic, diatomic, polyatomic) and with different expansion ratios. Which of these changes
the efficiency, and in which direction?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
print(f"{'f':>3} {'ratio':>7} {'N':>8}   efficiency")
print("-" * 40)
for dof in (3, 5, 6):
    for ratio in (1.2, 2.5, 8.0):
        eta = engines.carnot_cycle(
            N_PARTICLES, T_HOT, T_COLD, V_START, ratio, degrees_of_freedom=dof
        ).efficiency
        print(f"{dof:>3} {ratio:>7.1f} {N_PARTICLES:>8}   {eta:.12f}")

# And over four decades of engine size, at fixed gas and shape:
for n in (10, 1000, 100_000):
    eta = engines.carnot_cycle(n, T_HOT, T_COLD, V_START, 2.5).efficiency
    print(f"{3:>3} {2.5:>7.1f} {n:>8}   {eta:.12f}")

Twelve figures, every time. The gas does not matter; the size does not matter; the shape of
the loop does not matter. This is Carnot's theorem, and it is worth sitting with: the proof on
the module page never opens the engine, so nothing about the engine's insides can appear in
the answer.

The heats themselves are *not* the same — a bigger engine moves proportionally more energy.
It is their ratio that is fixed.

## Part 2 — Try to beat the bound

Now the interesting experiment. Build engines at random: any size, any expansion ratio, any
quality of thermal contact. See if any of them exceeds $1 - T_c/T_h$.

In [ ]:
efficiencies = []
for seed in range(200):
    engine = engines.random_two_reservoir_engine(np.random.default_rng(seed), T_HOT, T_COLD)
    efficiencies.append(engine.efficiency)

efficiencies = np.array(efficiencies)
bound = engines.carnot_efficiency(T_HOT, T_COLD)

plt.figure(figsize=(7, 4))
plt.hist(efficiencies, bins=30, color="#2563eb", alpha=0.75)
plt.axvline(bound, color="#dc2626", lw=2.2, label=f"Carnot bound = {bound:.3f}")
plt.xlabel("measured efficiency")
plt.ylabel("engines")
plt.legend()
plt.show()

print(f"engines built:            {efficiencies.size}")
print(f"best efficiency found:    {efficiencies.max():.6f}")
print(f"Carnot bound:             {bound:.6f}")
print(f"how many exceeded it:     {(efficiencies > bound).sum()}")

None of them. Try changing the seed range, the reservoir temperatures, the gas — the wall does
not move.

Be careful about what this shows. Two hundred engines respecting a bound is **not** a proof
that it cannot be beaten; no finite sample could be, and every engine here was built by code
that already implements the physics correctly. What the sweep really is, is a *falsification
test*: if you could make one exceed the bound, either the library or the derivation would be
wrong. Failing to break something you tried hard to break is evidence, not proof.

## Part 2b — Impossible machines, audited

The module page proves its key results by gluing machines together and reading the pair as a
single machine. Those arguments are pure bookkeeping, so the computer can keep the books. Each
machine below is a record of what it takes from the hot reservoir, from the cold reservoir,
and from outside as work — positive means *into the machine*. Adding the records is reading
the machines as one.

First the equivalence of the Kelvin and Clausius statements, both halves.

In [ ]:
def combine(*machines):
    # Read several machines as one: add up what each takes from each source.
    return {key: sum(m[key] for m in machines) for key in ("from_hot", "from_cold", "work_in")}


def verdict(machine):
    hot, cold, work = machine["from_hot"], machine["from_cold"], machine["work_in"]
    tol = 1e-9 * max(abs(hot), abs(cold), abs(work), 1.0)
    if abs(cold) < tol and hot > tol and work < -tol:
        return "turns one reservoir's heat entirely into work: a Kelvin violator"
    if abs(work) < tol and cold > tol and hot < -tol:
        return "moves heat from cold to hot with no work: a Clausius violator"
    return "allowed"


# (a) A Clausius violator C lifts 200 J from cold to hot and needs no work...
C = {"from_hot": -200.0, "from_cold": 200.0, "work_in": 0.0}
# ...beside an ordinary engine E: 500 J from the hot side, 200 J dumped, 300 J of work out.
E = {"from_hot": 500.0, "from_cold": -200.0, "work_in": -300.0}
print("C + E :", combine(C, E), "->", verdict(combine(C, E)))

# (b) A Kelvin violator K turns 300 J from the hot reservoir into 300 J of work...
K = {"from_hot": 300.0, "from_cold": 0.0, "work_in": -300.0}
# ...which drives an ordinary refrigerator F that lifts 400 J out of the cold side.
F = {"from_hot": -700.0, "from_cold": 400.0, "work_in": 300.0}
print("K + F :", combine(K, F), "->", verdict(combine(K, F)))

Now Carnot's proof, with a real reversible machine from the library in the role of $R$. The
hypothetical engine $X$ claims an efficiency of 0.6 between reservoirs whose Carnot bound is
0.5. Scale the library's reversible refrigerator so that it consumes exactly the work $X$
delivers, connect them, and read the pair.

In [ ]:
eta_x = 0.6
work = 300.0  # J that X delivers per cycle
X = {"from_hot": work / eta_x, "from_cold": -(work / eta_x - work), "work_in": -work}

reversed_r = engines.reversed_carnot_cycle(N_PARTICLES, T_HOT, T_COLD, V_START, 2.5)
scale = work / (-reversed_r.work_output)  # make R consume exactly X's work
R = {"from_hot": -scale * reversed_r.heat_rejected,
     "from_cold": scale * reversed_r.heat_absorbed,
     "work_in": work}


def rounded(machine):
    return {key: round(value, 6) for key, value in machine.items()}


print("X alone     :", rounded(X))
print("R backwards :", rounded(R))
print("X + R       :", rounded(combine(X, R)), "->", verdict(combine(X, R)))

The pair consumes no work and moves 100 J from cold to hot every cycle: the Clausius
statement's forbidden machine. So $X$ cannot exist — the proof, run as arithmetic. Change
`eta_x` to anything above 0.5 and the verdict stays the same; set it to exactly 0.5 and the
pair does nothing at all; below 0.5 the pair merely lets heat run downhill, which is allowed.

## Part 3 — Run it backwards

Every stroke of a reversible cycle can be reversed. Do that and the engine becomes a
refrigerator: work goes in, heat comes out of the cold side.

### Predict

The refrigerator below keeps a 275 K interior in a 298 K kitchen. Before running the next
cell: at best, how many joules of heat can it lift out of the interior per joule of
electricity — less than 1, about 1, or far more than 1?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
fridge = engines.reversed_carnot_cycle(N_PARTICLES, 298.0, 275.0, V_START, 2.5)

print(f"work consumed per cycle   = {-fridge.work_output:.4e} J")
print(f"heat lifted from the cold = {fridge.heat_absorbed:.4e} J")
print(f"coefficient of performance = {fridge.coefficient_of_performance:.4f}")
print(f"closed form T_c/(T_h-T_c)  = {engines.cop_refrigerator(298.0, 275.0):.4f}")
print()
print(f"as a heat pump, COP        = {engines.cop_heat_pump(298.0, 275.0):.4f}")
print(f"difference is exactly 1:     "
      f"{engines.cop_heat_pump(298.0, 275.0) - engines.cop_refrigerator(298.0, 275.0):.10f}")

# Asking a refrigerator for an efficiency is a category error, and the library says so.
try:
    _ = fridge.efficiency  # bound only so the access is not a bare expression
except ValueError as error:
    print(f"\nasking for its efficiency: {error}")

### Does the fridge break the second law?

It genuinely lowers the entropy of the food inside it. Run the bookkeeping and see where the
compensation comes from — this is the experiment that kills the "a fridge violates the second
law" reading, because both entropies are computed, not asserted.

The interior is held at 275 K, so it acts as a reservoir: its entropy change is just the
heat lifted divided by its temperature. (A body whose temperature changed would need
$C \ln(T_2/T_1)$ instead; the module page derives it.)

In [ ]:
T_COLD_IN, T_KITCHEN = 275.0, 298.0
heat_lifted = 1000.0  # J removed from the food

for label, cop in [("a real fridge", 3.2), ("the best possible fridge",
                                            engines.cop_refrigerator(T_KITCHEN, T_COLD_IN))]:
    work = heat_lifted / cop
    dumped = heat_lifted + work           # first law: everything lifted, plus the work
    food = -heat_lifted / T_COLD_IN       # the food's entropy really does fall
    kitchen = dumped / T_KITCHEN          # the kitchen's rises
    print(f"{label} (COP {cop:.2f}):")
    print(f"    food    dS = {food:+.4f} J/K")
    print(f"    kitchen dS = {kitchen:+.4f} J/K")
    total = food + kitchen
    # The reversible fridge sits exactly at zero, so what comes back there is rounding of
    # either sign. A bare "-0.0000" would read as precisely the violation this cell rules
    # out, so display it as the zero it is.
    shown = 0.0 if abs(total) < 1e-9 * abs(food) else total
    print(f"    total   dS = {shown:+.4f} J/K")
    assert food < 0.0                     # the misconception's premise is TRUE
    assert total > -1e-9 * abs(food)      # ...and its conclusion still does not follow
    print()

# The library's own reversible refrigerator, built stroke by stroke, gives the same verdict:
best = engines.reversed_carnot_cycle(N_PARTICLES, T_KITCHEN, T_COLD_IN, V_START, 2.5)
print(f"library reversible fridge: entropy produced {best.entropy_produced:+.1e} J/K "
      f"against a scale of {best.entropy_scale:.1e} J/K - zero to rounding")
print()
print("The premise holds and the conclusion fails: the second law constrains the total.")
print("Only the reversible fridge reaches zero, and nothing gets below it.")

A coefficient of performance of about 12 is not a violation of anything. Nothing is being
*converted* here — energy is being **moved**, and moving heat uphill costs less than the
amount moved. That is also why a heat pump heats a house for a fraction of what a resistive
heater costs.

## Part 4 — Where the lost work goes

A real engine cannot touch its reservoirs at exactly their temperatures: heat will not cross a
zero temperature difference at a finite rate. Give the gas a gap at each end and watch two
things move together.

### Predict

As the gap at each end grows from 0 to 100 K, how does the efficiency fall — in a straight
line, faster, slower? And the entropy produced per cycle: does it stay at zero, jump to a
constant, or grow with the gap?

**Your prediction:**

*(write here before running the next cell)*

In [ ]:
gaps = np.linspace(0.0, 100.0, 26)
etas, produced = [], []
for gap in gaps:
    engine = engines.endoreversible_cycle(
        N_PARTICLES, T_HOT, T_COLD, V_START, 2.5, hot_gap=float(gap), cold_gap=float(gap)
    )
    etas.append(engine.efficiency)
    produced.append(engine.entropy_produced)

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
left.plot(gaps, etas, lw=2.2, color="#2563eb")
left.axhline(engines.carnot_efficiency(T_HOT, T_COLD), ls="--", color="#dc2626",
             label="Carnot bound")
left.set_xlabel("temperature gap at each end (K)")
left.set_ylabel("efficiency")
left.legend()

right.plot(gaps, produced, lw=2.2, color="#d97706")
right.set_xlabel("temperature gap at each end (K)")
right.set_ylabel("entropy produced per cycle (J/K)")
plt.tight_layout()
plt.show()

The efficiency falls and the entropy produced rises, together. They are not two effects but
one. Compare the gapped engine with a reversible one drawing the same heat $Q_h$ from the same
hot reservoir: the reversible one delivers $Q_h (1 - T_c/T_h)$, the gapped one $Q_h - Q_c$, and
the shortfall is

$$
W_{\text{lost}} = Q_c - \frac{T_c}{T_h} Q_h
= T_c \left(\frac{Q_c}{T_c} - \frac{Q_h}{T_h}\right)
= T_c\, S_{\text{gen}} .
$$

This is the Gouy–Stodola theorem, derived on the module page in the section on entropy
produced. The next cell checks it numerically, and then runs five checks that also live in the
project's test suite:

1. the loop closes — four strokes computed from four independent closed forms return to the
   starting state;
2. the first law closes around the loop;
3. the reversible cycle produces no entropy, judged against the size of one term of the
   Clausius sum, because the exact answer is zero and rounding has either sign;
4. the gapped cycle produces a strictly positive amount;
5. the area enclosed in the $P$–$V$ plane, integrated numerically, equals the work computed
   from the closed forms.

In [ ]:
engine = engines.endoreversible_cycle(N_PARTICLES, T_HOT, T_COLD, V_START, 2.5, 40.0, 40.0)
perfect = engines.carnot_cycle(N_PARTICLES, T_HOT, T_COLD, V_START, 2.5)

# Scale both to the same heat absorbed so the comparison is fair.
scaled_perfect_work = perfect.efficiency * engine.heat_absorbed
lost = scaled_perfect_work - engine.work_output

print(f"work delivered by the real engine   = {engine.work_output:.6e} J")
print(f"a perfect engine on the same heat   = {scaled_perfect_work:.6e} J")
print(f"work lost                           = {lost:.6e} J")
print(f"T_c * entropy produced              = {T_COLD * engine.entropy_produced:.6e} J")
print(f"relative difference                 = "
      f"{relative_error(lost, T_COLD * engine.entropy_produced):.2e}")

# --- the checks that also live in the project's test suite ---
scale = abs(cycle.strokes[0].process.start.internal_energy)

# 1. The loop closes: four independent closed forms must agree.
assert abs(cycle.internal_energy_drift) / scale < 1e-12

# 2. The first law closes around the loop.
assert abs(cycle.first_law_residual) / scale < 1e-12

# 3. A reversible cycle produces no entropy.
assert abs(perfect.entropy_produced) / perfect.entropy_scale < 1e-12

# 4. An irreversible one produces a strictly positive amount.
assert engine.entropy_produced / engine.entropy_scale > 1e-6

# 5. The area enclosed really is the work delivered (quadrature vs closed form).
assert relative_error(perfect.enclosed_area, perfect.work_output) < 1e-4

print("\nall five checks passed")

## Part 5 — Entropy, the Clausius way

The free expansion of module 06 exchanges no heat and does no work. Put it into a cycle: let
the gas expand freely to twice its volume, then push it back along the isotherm in contact
with a reservoir at its own temperature.

### Predict

Two numbers, before you run the cell: the gas's entropy change over the free expansion, and
the cycle's Clausius sum $\oint \delta Q / T_{\text{res}}$. Zero, positive or negative?

In [ ]:
start = processes.EquilibriumState.from_temperature(N_PARTICLES, T_COLD, V_START)
free = processes.free_expansion(start, 2 * V_START)
back = processes.isothermal(free.end, V_START)

# One reservoir only. CycleResult asks for a hot/cold pair for its Carnot bound, which is not
# used here, so the hot one is a placeholder that no stroke touches.
loop = engines.CycleResult(
    label="free expansion and back",
    strokes=(engines.Stroke(free), engines.Stroke(back, T_COLD)),
    t_hot=T_HOT,
    t_cold=T_COLD,
)
n_kb_ln2 = N_PARTICLES * K_B * np.log(2.0)
round_trip = sum(engines.entropy_change_of_gas(s.process) for s in loop.strokes)

print(f"heat into the gas, free expansion   = {free.heat:.1e} J")
print(f"gas entropy change, free expansion  = {engines.entropy_change_of_gas(free):.6e} J/K")
print(f"N k_B ln 2                          = {n_kb_ln2:.6e} J/K")
print(f"gas entropy change, whole loop      = {round_trip:+.1e} J/K")
print(f"Clausius sum, whole loop            = {loop.clausius_sum:.6e} J/K")
print(f"entropy produced                    = {loop.entropy_produced:.6e} J/K")

No heat crossed during the free expansion, yet the gas's entropy rose by exactly
$N k_B \ln 2$ — the value a reversible isotherm gives between the same two states. Entropy
belongs to the state; the reversible path is only the instrument for computing it. The
Clausius sum has the same size and the opposite sign: the reservoir received the
recompression's heat and nothing ever gave it back. That is the entropy the free expansion produced.

Module 08 gets the same $N k_B \ln 2$ by a completely different route: counting the positions
open to $N$ particles when their volume doubles. The two definitions of entropy agreeing is not
a coincidence.

## Part 6 — A real engine: the Otto cycle

The idealised petrol engine: compress the air adiabatically by a factor $r$, burn the fuel at
constant volume, expand adiabatically, exhaust at constant volume. Every stroke is reversible.
Compare its efficiency with $1 - r^{1-\gamma}$, and with the Carnot bound between the coldest
and hottest temperatures the air reaches.

In [ ]:
gamma_air = processes.gamma_from_dof(5)

print(f"{'r':>5}{'Otto':>10}{'1 - r^(1-g)':>13}{'T_max (K)':>12}{'Carnot, extremes':>19}")
for ratio in (4.0, 8.0, 12.0, 20.0):
    otto = engines.otto_cycle(N_PARTICLES, 300.0, V_START, ratio, heat_input=2.0e-17)
    print(f"{ratio:>5.0f}{otto.efficiency:>10.4f}"
          f"{engines.otto_efficiency(ratio, gamma_air):>13.4f}"
          f"{otto.t_hot:>12.0f}{otto.carnot_bound:>19.4f}")

# No reservoir temperature describes a heat stroke whose gas temperature is changing, and the
# library would rather refuse than invent one.
try:
    _ = otto.clausius_sum  # bound only so the access is not a bare expression
except ValueError as error:
    print(f"\nasking for its Clausius sum: {error}")

Every row falls short of the Carnot bound between its own extremes, though no stroke is
irreversible. Worked example 5 on the module page explains why: the Otto engine takes its heat
in while the air is still warming up and gives it out while the air is still cooling down, so
none of its heat crosses the full temperature span. Slice the cycle with adiabats and every
slice is a small Carnot engine working across a narrower span.

The explorer lets you set the compression ratio and the peak temperature the burn reaches.
Press **Run Interact** after moving the sliders.

In [ ]:
import ipywidgets as widgets


def explore_otto(compression_ratio=8.0, peak_temperature=2000.0):
    gamma = processes.gamma_from_dof(5)
    after_compression = float(processes.adiabatic_final_temperature(
        300.0, V_START, V_START / compression_ratio, gamma))
    if peak_temperature <= after_compression:
        print(f"compression alone reaches {after_compression:.0f} K - "
              f"set the peak temperature above it")
        return
    heat = processes.heat_capacity_constant_volume(N_PARTICLES, 5) * (
        peak_temperature - after_compression)
    otto = engines.otto_cycle(N_PARTICLES, 300.0, V_START, compression_ratio, heat)

    fig, (plane, bars) = plt.subplots(1, 2, figsize=(11, 3.8),
                                      gridspec_kw={"width_ratios": [1.4, 1]})
    for stroke, colour in zip(otto.strokes, ["#94a3b8", "#dc2626", "#94a3b8", "#2563eb"],
                              strict=True):
        path = stroke.process.quasistatic_path
        plane.plot(path.volumes * 1e3, path.pressures, lw=2.2, color=colour)
    plane.set_xlabel("volume (L)")
    plane.set_ylabel("pressure (Pa)")

    bars.bar(["Otto", "Carnot, same extremes"], [otto.efficiency, otto.carnot_bound],
             color=["#2563eb", "#dc2626"])
    bars.set_ylim(0, 1)
    bars.set_ylabel("efficiency")
    plt.tight_layout()
    plt.show()

    print(f"after compression   {after_compression:.0f} K")
    print(f"after expansion     {otto.strokes[2].process.end.temperature:.0f} K")
    print(f"Otto efficiency     {otto.efficiency:.4f}")
    print(f"Carnot bound        {otto.carnot_bound:.4f}  (300 K to {peak_temperature:.0f} K)")


widgets.interact_manual(
    explore_otto,
    compression_ratio=widgets.FloatSlider(min=2.0, max=20.0, step=0.5, value=8.0),
    peak_temperature=widgets.FloatSlider(min=800.0, max=3000.0, step=50.0, value=2000.0),
);

## Part 7 — Explore it yourself

The sliders let you vary the reservoirs and the quality of the thermal contact. Two
experiments worth doing:

1. Bring the two reservoir temperatures close together and watch the efficiency collapse.
   This is why low-grade waste heat is nearly worthless however much of it there is.
2. Hold the temperatures fixed and open up the gaps. Watch how much efficiency you lose for
   the privilege of running at a finite rate.

Press **Run Interact** after moving the sliders — the callback redraws two panels.

In [ ]:
import ipywidgets as widgets


def explore(t_hot=600.0, t_cold=300.0, gap=20.0, expansion_ratio=2.5):
    span = t_hot - t_cold
    gap = min(gap, 0.45 * span)  # keep the working temperatures from crossing
    engine = engines.endoreversible_cycle(
        N_PARTICLES, t_hot, t_cold, V_START, expansion_ratio,
        hot_gap=gap, cold_gap=gap,
    )
    bound = engines.carnot_efficiency(t_hot, t_cold)

    fig, (plane, bars) = plt.subplots(1, 2, figsize=(11, 3.8),
                                      gridspec_kw={"width_ratios": [1.4, 1]})
    for stroke, colour in zip(engine.strokes,
                              ["#dc2626", "#94a3b8", "#2563eb", "#94a3b8"],
                              strict=True):
        path = stroke.process.quasistatic_path
        plane.plot(path.volumes * 1e3, path.pressures, lw=2.2, color=colour)
    plane.set_xlabel("volume (L)")
    plane.set_ylabel("pressure (Pa)")

    bars.bar(["this engine", "Carnot bound"], [engine.efficiency, bound],
             color=["#2563eb", "#dc2626"])
    bars.set_ylim(0, 1)
    bars.set_ylabel("efficiency")
    plt.tight_layout()
    plt.show()

    print(f"efficiency        {engine.efficiency:.4f}")
    print(f"Carnot bound      {bound:.4f}")
    print(f"entropy produced  {engine.entropy_produced:.3e} J/K per cycle")


widgets.interact_manual(
    explore,
    t_hot=widgets.FloatSlider(min=350, max=1200, step=25, value=600),
    t_cold=widgets.FloatSlider(min=200, max=340, step=5, value=300),
    gap=widgets.FloatSlider(min=0, max=100, step=5, value=20),
    expansion_ratio=widgets.FloatSlider(min=1.2, max=8.0, step=0.2, value=2.5),
);

## Check your understanding

Run the cell below for the auto-graded quiz. The same questions, with written explanations
for every option, are on the module page.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "07-second-law.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## Before you leave

Write a few sentences on each, in the cell below.

1. What did you predict that turned out to be wrong, and what specifically was the flaw in
   your reasoning?
2. You tried to beat the Carnot bound and failed. Explain why that failure is not a proof of
   Carnot's theorem, and say what would count as one.
3. A colleague says the second law is "really just about friction and losses". Give them the
   one-sentence correction that a perfect, frictionless engine would still respect.
4. Is doing a process infinitely slowly enough to make it reversible? Give an example that
   settles it.
5. The free expansion received no heat, yet the gas's entropy rose. Explain how both can be
   true at once.

**Your answers:**

1.
2.
3.
4.
5.